In [11]:
import pandas as pd
import networkx as nx
import ast
import itertools
import os

# --- Reconstrução do Grafo G ---
# (Este código é o mesmo de antes, caso você tenha reiniciado o notebook)
# Se a variável 'G' já existe na memória, pule esta seção.
if 'G' not in locals():
    print("Recriando o grafo G...")

    # --- Passo 1: Carregar e Processar o DataFrame ---
    csv_file = 'weapons.csv'
    if not os.path.exists(csv_file):
        print(f"ERRO: Arquivo '{csv_file}' não encontrado. Faça o upload primeiro.")
        # Se estiver no Colab, o script para aqui
        exit()

    df_weapons = pd.read_csv(csv_file)
    scaling_map = {'S': 5, 'A': 4, 'B': 3, 'C': 2, 'D': 1, 'E': 0, '-': -1}

    def get_primary_scaling(scales_str):
        try:
            scales_list = ast.literal_eval(scales_str)
            if not scales_list: return []
            max_scaling_val = max(scaling_map.get(s.get('scaling', '-'), -1) for s in scales_list)
            return [s['name'] for s in scales_list if scaling_map.get(s.get('scaling', '-'), -1) == max_scaling_val]
        except (ValueError, SyntaxError):
            return []

    df_weapons['parsed_scalesWith'] = df_weapons['scalesWith'].apply(get_primary_scaling)

    # --- Passo 2: Construir o Grafo com NetworkX ---
    G = nx.Graph()
    for _, weapon in df_weapons.iterrows():
        # Armazenar o escalonamento como uma string para ser mais fácil de ler
        G.add_node(
            weapon['name'],
            category=weapon['category'],
            weight=weapon['weight'],
            primary_scaling=str(weapon['parsed_scalesWith'])
        )

    weapon_scales_map = {row['name']: set(row['parsed_scalesWith']) for _, row in df_weapons.iterrows()}

    for weapon_a, weapon_b in itertools.combinations(weapon_scales_map.keys(), 2):
        if weapon_scales_map[weapon_a].intersection(weapon_scales_map[weapon_b]):
            G.add_edge(weapon_a, weapon_b)

    print("Grafo G recriado.")
# --- Fim da Reconstrução ---

print("\n--- Calculando Centralidade de Grau (Degree Centrality) ---")

# G.degree() retorna uma lista de tuplas (nó, grau)
degree_list = list(G.degree())

# Ordenar a lista pelo grau (o segundo item da tupla, index 1)
sorted_degree = sorted(degree_list, key=lambda item: item[1], reverse=True)

# --- NOVO CÓDIGO AQUI ---
# Vamos "enriquecer" a lista do Top 10 com o atributo de escalonamento

print("\n--- Top 10 Armas por Grau (com Motivo da Conexão) ---")
print("Este ranking mostra as armas 'hub' e o(s) atributo(s) que causam tantas conexões.")

# Criar uma nova lista de dados para o DataFrame
enriched_top_10 = []
for weapon_name, degree in sorted_degree[:10]:
    # Buscar o atributo 'primary_scaling' que salvamos no nó do grafo
    scaling_info = G.nodes[weapon_name]['primary_scaling']
    enriched_top_10.append([weapon_name, degree, scaling_info])

# Criar o DataFrame com a nova coluna
top_10_df = pd.DataFrame(enriched_top_10, columns=['Arma', 'Grau (Nº de Conexões)', 'Escalonamento Primário (O Motivo)'])
print(top_10_df.to_markdown(index=False, numalign="left", stralign="left"))


# --- Cálculo das Outras Centralidades (Sem Alteração) ---

print("\n--- Calculando Outras Centralidades (Intermediação e Proximidade) ---")
print("(Isso pode demorar um pouco...)")

# Calcular Centralidade de Intermediação (Betweenness)
betweenness_cen = nx.betweenness_centrality(G)
sorted_betweenness = sorted(betweenness_cen.items(), key=lambda item: item[1], reverse=True)

# Calcular Centralidade de Proximidade (Closeness)
closeness_cen = nx.closeness_centrality(G)
sorted_closeness = sorted(closeness_cen.items(), key=lambda item: item[1], reverse=True)

print("\n--- Top 10 Armas por Centralidade de Intermediação (Betweenness) ---")
print("Este ranking mostra as armas que mais servem como 'pontes' entre grupos.")
print("Interpretação: Armas 'híbridas' que conectam diferentes tipos de builds.")
top_10_betweenness_df = pd.DataFrame(sorted_betweenness[:10], columns=['Arma', 'Centralidade de Intermediação'])
top_10_betweenness_df['Centralidade de Intermediação'] = top_10_betweenness_df['Centralidade de Intermediação'].round(4)
print(top_10_betweenness_df.to_markdown(index=False, numalign="left", stralign="left"))


print("\n--- Top 10 Armas por Centralidade de Proximidade (Closeness) ---")
print("Este ranking mostra as armas mais 'centrais' da rede (menor caminho médio).")
print("Interpretação: Armas 'representativas' de onde é mais fácil 'pivotar' para outras armas.")
top_10_closeness_df = pd.DataFrame(sorted_closeness[:10], columns=['Arma', 'Centralidade de Proximidade'])
top_10_closeness_df['Centralidade de Proximidade'] = top_10_closeness_df['Centralidade de Proximidade'].round(4)
print(top_10_closeness_df.to_markdown(index=False, numalign="left", stralign="left"))

print("\nAnálise de centralidade concluída.")


--- Calculando Centralidade de Grau (Degree Centrality) ---

--- Top 10 Armas por Grau (com Motivo da Conexão) ---
Este ranking mostra as armas 'hub' e o(s) atributo(s) que causam tantas conexões.
| Arma                   | Grau (Nº de Conexões)   | Escalonamento Primário (O Motivo)   |
|:-----------------------|:------------------------|:------------------------------------|
| Troll Knight's Sword   | 276                     | ['Str', 'Dex', 'Int']               |
| Crystal Knife          | 276                     | ['Str', 'Dex', 'Int']               |
| Loretta's War Sickle   | 276                     | ['Str', 'Dex', 'Int']               |
| Crystal Spear          | 276                     | ['Str', 'Dex', 'Int']               |
| Rotten Crystal Spear   | 276                     | ['Str', 'Dex', 'Int']               |
| Carian Knight's Sword  | 276                     | ['Str', 'Dex', 'Int']               |
| Godslayer's Greatsword | 271                     | ['Str', 'Dex', 'Fai']